# QASPER Data Download and Preparation

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
!pip -q install -U datasets huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
data_files = {
    "train":      "hf://datasets/allenai/qasper@refs/convert/parquet/qasper/train/*.parquet",
    "validation": "hf://datasets/allenai/qasper@refs/convert/parquet/qasper/validation/*.parquet",
    "test":       "hf://datasets/allenai/qasper@refs/convert/parquet/qasper/test/*.parquet",
}
ds = load_dataset("parquet", data_files=data_files)
ds

In [ ]:
save_dir = "/Volumes/main/default/thesis_project/data/qasper/qasper_datasetdict"
ds.save_to_disk(save_dir)
print("Saved to:", save_dir)

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_from_disk

qasper_path = "/Volumes/main/default/thesis_project/data/qasper/qasper_datasetdict"
dataset = load_from_disk(qasper_path)
print(dataset)


In [ ]:
from pathlib import Path
from datasets import load_from_disk
QASPER_DIR = Path("/Volumes/main/default/thesis_project/data/qasper/qasper_datasetdict")  # TODO: change this to your path
ds = load_from_disk(str(QASPER_DIR))
print(ds)

In [ ]:
# 2 Utility
import pandas as pd
import numpy as np
import json
from typing import List, Dict, Any, Optional

In [ ]:
def non_empty(s):
    return isinstance(s, str) and s.strip() != ""


In [ ]:
def normalize_answer_from_one_annotation(a: Dict[str, Any]) -> Optional[str]:
    """
    Build a single textual gold answer from one annotation:
      - prefer free_form_answer
      - else yes/no -> "Yes"/"No"
      - else join extractive_spans (unique, space-joined)
    Skip if marked unanswerable.
    """
    if not isinstance(a, dict):
        return None
    if a.get("unanswerable", False):
        return None
    # free-form
    ffa = a.get("free_form_answer")
    if non_empty(ffa):
        return ffa.strip()
    # yes/no (boolean)
    if "yes_no" in a and a["yes_no"] is not None:
        if isinstance(a["yes_no"], bool):
            return "Yes" if a["yes_no"] else "No"
    # extractive spans
    spans = a.get("extractive_spans") or []
    spans = [s.strip() for s in spans if non_empty(s)]
    if spans:
        # keep unique order
        seen, uniq = set(), []
        for s in spans:
            if s not in seen:
                seen.add(s); uniq.append(s)
        return " ".join(uniq)
    return None

In [ ]:
def pick_gold_answer(answers: List[Dict[str, Any]]) -> Optional[str]:
    """
    There may be multiple annotations (from different workers).
    Strategy:
      1) all candidate answers via normalize_answer_from_one_annotation
      2) choose the most common non-empty (majority); tie -> first
    """
    cands = []
    for a in answers or []:
        s = normalize_answer_from_one_annotation(a)
        if non_empty(s):
            cands.append(s)
    if not cands:
        return None
    # majority vote on strings (exact match)
    vc = pd.Series(cands).value_counts()
    return vc.index[0]

In [ ]:
def collect_oracle_evidence_from_one_annotation(a: Dict[str, Any]) -> List[str]:
    """
    Prefer sentence-level highlighted_evidence, otherwise paragraph-level evidence.
    Filter out figure/table markers starting with 'FLOAT SELECTED'.
    """
    ev = []
    if not isinstance(a, dict):
        return ev
    # sentence-level first
    hi = a.get("highlighted_evidence") or []
    hi = [x for x in hi if non_empty(x) and not str(x).startswith("FLOAT SELECTED")]
    if hi:
        ev.extend(hi)
    # fallback: paragraph-level
    if not ev:
        para = a.get("evidence") or []
        para = [x for x in para if non_empty(x) and not str(x).startswith("FLOAT SELECTED")]
        ev.extend(para)
    return ev

In [ ]:
def pick_oracle_evidence(answers: List[Dict[str, Any]], max_paras=20) -> Optional[str]:
    """
    Merge evidence from multiple annotations:
      - collect all sentence evidence; if none, collect paragraph evidence
      - de-duplicate by text, keep order
      - join with newline
    """
    gathered = []
    for a in answers or []:
        gathered.extend(collect_oracle_evidence_from_one_annotation(a))
    # de-duplicate keeping order
    seen, uniq = set(), []
    for t in gathered:
        s = t.strip()
        if s and s not in seen:
            seen.add(s); uniq.append(s)
    if not uniq:
        return None
    # cut very long contexts
    if max_paras and len(uniq) > max_paras:
        uniq = uniq[:max_paras]
    return "\n".join(uniq)

In [ ]:
def iter_rows_from_split(split_name: str, dataset):
    """
    Expand a QASPER split into per-question rows:
      For each paper row: multiple questions in row["qas"]["question"],
      and for each question we have list of answer annotations in row["qas"]["answers"][i]
    Output unified rows with question_id and paper_id.
    """
    for paper in dataset:
        paper_id = paper["id"]
        qas = paper["qas"]
        questions = qas.get("question") or []
        question_ids = qas.get("question_id") or list(range(len(questions)))
        answers_list = qas.get("answers") or []
        # iterate by index
        for i, q in enumerate(questions):
            q_text = (q or "").strip()
            q_id = question_ids[i] if i < len(question_ids) else f"{paper_id}#q{i}"
            ann = answers_list[i] if i < len(answers_list) else {}
            # ann is expected like {'answers':[ {...}, {...} ], ...}
            annotations = ann.get("answer") or []
            # build gold and evidence
            gold = pick_gold_answer(annotations)
            evidence = pick_oracle_evidence(annotations)
            yield {
                "split": split_name,
                "paper_id": paper_id,
                "question_id": q_id,
                "title": paper.get("title"),
                "question": q_text,
                "gold_answer": gold or "",
                "oracle_evidence": evidence or "",
            }

In [ ]:
# 3 Build raw and clean DataFrames
rows = []
for sp in ["train","validation","test"]:
    if sp in ds:
        rows.extend(iter_rows_from_split(sp, ds[sp]))

In [ ]:
raw_df = pd.DataFrame(rows)
print("Raw rows:", raw_df.shape)

In [ ]:
# Integrity stats
stats = {
    "total_rows": int(len(raw_df)),
    "empty_question": int((~raw_df["question"].apply(non_empty)).sum()),
    "empty_gold_answer": int((~raw_df["gold_answer"].apply(non_empty)).sum()),
    "empty_oracle_evidence": int((~raw_df["oracle_evidence"].apply(non_empty)).sum()),
}
print("Integrity:", stats)

In [ ]:
# Cleaning: keep rows that have BOTH a gold answer AND non-empty evidence
clean_df = raw_df.copy()
for col in ["question","gold_answer","oracle_evidence"]:
    clean_df[col] = clean_df[col].fillna("").apply(lambda s: s.strip())

mask = clean_df["gold_answer"].apply(non_empty) & clean_df["oracle_evidence"].apply(non_empty)
clean_df = clean_df[mask].drop_duplicates(
    subset=["paper_id","question","gold_answer","oracle_evidence"]
).reset_index(drop=True)

print("Rows after cleaning:", len(clean_df))
print(clean_df["split"].value_counts())

In [ ]:
# 4 Compact preview
pd.set_option("display.max_colwidth", 180)
preview_cols = ["split","paper_id","question_id","question","gold_answer"]
display(clean_df[preview_cols].head(10))

In [ ]:
example = clean_df.sample(1, random_state=0)[
    ["split","question","gold_answer","oracle_evidence"]
]
print("\n=== One full example (truncated) ===")
display(example)

In [ ]:
# 5 Export to a timestamped folder on Drive (same style as GovReport-QS)
from datetime import datetime
from zoneinfo import ZoneInfo  # Python 3.9+
SAVE_BASE = Path("/Volumes/main/default/thesis_project/data/qasper/qasper_datasetdict")

In [ ]:
now_cph = datetime.now(ZoneInfo("Europe/Copenhagen"))  # or datetime.now()
stamp = now_cph.strftime("%Y%m%d_%H%M%S")
OUT_DIR = SAVE_BASE / f"processed_{stamp}"
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
clean_df.to_parquet(OUT_DIR / "qasper_all_qa_evidence.parquet", index=False)
clean_df.to_csv(OUT_DIR / "qasper_all_qa_evidence.csv", index=False)

In [ ]:
# per split
for sp in sorted(clean_df["split"].dropna().unique()):
    df_sp = clean_df[clean_df["split"]==sp]
    df_sp.to_parquet(OUT_DIR / f"qasper_{sp}_qa_evidence.parquet", index=False)
    df_sp.to_csv(OUT_DIR / f"qasper_{sp}_qa_evidence.csv", index=False)


In [ ]:
print("\nSaved to:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)